In [ ]:
!pip install uv
!uv pip install  -r requirements.txt 

Cell 1: Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import xarray as xr
from scipy.spatial import cKDTree
import pystac_client
import planetary_computer as pc
from tqdm import tqdm
import base64
from IPython.display import HTML, display

# Dask client for parallel processing
from dask.distributed import Client

# Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("Libraries loaded successfully!")

Cell 2: Load Unique Samples for Extraction

In [ ]:
# Load the unique sample list created in Notebook 1
df_samples = pd.read_csv('unique_samples_for_extraction.csv')

print("="*80)
print("UNIQUE SAMPLES LOADED")
print("="*80)
print(f"\nTotal samples to extract: {len(df_samples):,}")
print(f"Columns: {list(df_samples.columns)}")

# Parse dates
df_samples['Sample Date'] = pd.to_datetime(df_samples['Sample Date'])

print(f"\nDate range:")
print(f"  Earliest: {df_samples['Sample Date'].min()}")
print(f"  Latest: {df_samples['Sample Date'].max()}")

print(f"\nFirst few rows:")
display(df_samples.head())

Cell 3: Define TerraClimate Variables & Extraction Function

In [ ]:
# TerraClimate variables to extract (same 14 as your old notebook)
tc_variables = [
    'pet',   # Potential evapotranspiration (mm)
    'aet',   # Actual evapotranspiration (mm)
    'ppt',   # Precipitation (mm)
    'tmax',  # Maximum temperature (°C)
    'tmin',  # Minimum temperature (°C)
    'q',     # Runoff (mm)
    'soil',  # Soil moisture (mm)
    'swe',   # Snow water equivalent (mm)
    'def',   # Climate water deficit (mm)
    'pdsi',  # Palmer Drought Severity Index
    'vap',   # Vapor pressure (kPa)
    'vpd',   # Vapor pressure deficit (kPa)
    'ws',    # Wind speed (m/s)
    'srad'   # Solar radiation (W/m²)
]

print("="*80)
print("TERRACLIMATE VARIABLES TO EXTRACT")
print("="*80)
print(f"\nTotal variables: {len(tc_variables)}")
for i, var in enumerate(tc_variables, 1):
    print(f"  {i:2d}. {var}")

Cell 4: Defining the terraclimate Extraction Functions

In [ ]:
def filterg_all(ds, variables, refresh_every=10):
    """Filter all variables with periodic token refresh"""
    ds_2011_2015 = ds[variables].sel(time=slice("2011-01-01", "2015-12-31"))
    df_all_append = []

    for i in tqdm(range(len(ds_2011_2015.time)), desc="Filtering TerraClimate data"):
        
        # Refresh token every N iterations to prevent expiry
        if i > 0 and i % refresh_every == 0:
            print(f"\nRefreshing Planetary Computer token at iteration {i}...")
            catalog = pystac_client.Client.open(
                "https://planetarycomputer.microsoft.com/api/stac/v1",
                modifier=pc.sign_inplace)
            collection = catalog.get_collection("terraclimate")
            asset = collection.assets["zarr-abfs"]
            ds_refreshed = xr.open_dataset(
                asset.href, **asset.extra_fields["xarray:open_kwargs"])
            ds_refreshed = ds_refreshed.drop('crs', dim=None)
            mask_lon = (ds_refreshed.lon >= 14.97) & (ds_refreshed.lon <= 32.79)
            mask_lat = (ds_refreshed.lat >= -35.18) & (ds_refreshed.lat <= -21.72)
            ds_refreshed = ds_refreshed.where(mask_lon & mask_lat, drop=True)
            ds_2011_2015 = ds_refreshed[variables].sel(time=slice("2011-01-01", "2015-12-31"))
            print("Token refreshed!")

        df_time = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_filtered = df_time[
            (df_time['lat'] > -35.18) & (df_time['lat'] < -21.72) &
            (df_time['lon'] > 14.97) & (df_time['lon'] < 32.79)
        ]
        df_all_append.append(df_filtered)

    df_final = pd.concat(df_all_append, ignore_index=True)
    col_mapping = {"lat": "Latitude", "lon": "Longitude", "time": "Sample Date"}
    df_final = df_final.rename(columns=col_mapping)
    df_final['Sample Date'] = df_final['Sample Date'].astype(str)

    print(f"\nFiltering completed for: {variables}")
    return df_final


def assign_nearest_climate_all(sa_df, climate_df, variables):
    """Assign nearest climate values to sample points"""
    sa_df = sa_df.copy().reset_index(drop=True)
    climate_df = climate_df.copy()

    # Parse dates
    sa_df['Sample Date'] = pd.to_datetime(sa_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')

    # Build KD-tree on climate_df coordinates
    print("Building KD-tree...")
    sa_coords = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    # Get nearest points
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df['nearest_lat'] = nearest_points['Latitude'].values
    sa_df['nearest_lon'] = nearest_points['Longitude'].values

    # Map all variables
    print("Mapping climate values to samples...")
    results = {var: [] for var in variables}

    for i in tqdm(range(len(sa_df)), desc="Assigning climate values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']

        # Filter climate_df to this grid point
        subset = climate_df[
            (climate_df['Latitude'] == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]

        if subset.empty:
            for var in variables:
                results[var].append(np.nan)
            continue

        # Find nearest date
        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()

        # Grab all variables for this row
        for var in variables:
            results[var].append(subset.loc[nearest_idx, var])

    # Add all variables to sa_df
    for var in variables:
        sa_df[var] = results[var]

    sa_df = sa_df.drop(columns=['nearest_lat', 'nearest_lon'])

    print(f"\nDone! Shape: {sa_df.shape}")
    print(f"\nNaN count per variable:")
    print(sa_df[variables].isna().sum())
    return sa_df


Cell 6: Initialize Planetary Computer & Load TerraClimate Dataset

In [ ]:
print("="*80)
print("LOADING TERRACLIMATE DATASET FROM PLANETARY COMPUTER")
print("="*80)

# Initialize Dask client
client = Client(memory_limit='4GB', processes=False)
print(f"Dask client initialized: {client}")

# Connect to Planetary Computer STAC catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace)

# Get TerraClimate collection
collection = catalog.get_collection("terraclimate")
asset = collection.assets["zarr-abfs"]

# Load dataset
print("\nLoading TerraClimate dataset...")
ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
ds = ds.drop('crs', dim=None)
ds = ds.sel(time=slice("2011-01-01", "2015-12-31"))

# Trim to South Africa bounding box
print("Trimming to South Africa bounding box...")
mask_lon = (ds.lon >= 14.97) & (ds.lon <= 32.79)
mask_lat = (ds.lat >= -35.18) & (ds.lat <= -21.72)
ds = ds.where(mask_lon & mask_lat, drop=True)

print("\n✅ Dataset loaded and trimmed!")
print(ds)

Cell 7: Extract TerraClimate Data for All Samples


In [ ]:
print("="*80)
print("EXTRACTING TERRACLIMATE DATA")
print("="*80)
print(f"\nThis will extract {len(tc_variables)} variables for {len(df_samples):,} samples")

# Filter all 14 variables in one pass
tc_all_filtered = filterg_all(ds, tc_variables, refresh_every=10)

print(f"\nFiltered climate data shape: {tc_all_filtered.shape}")
print(f"Filtered data date range: {tc_all_filtered['Sample Date'].min()} to {tc_all_filtered['Sample Date'].max()}")

Cell 8: Assign Climate Values to Sample Points

In [ ]:
print("="*80)
print("ASSIGNING CLIMATE VALUES TO SAMPLE POINTS")
print("="*80)

# Assign nearest climate values for all variables
terraclimate_df = assign_nearest_climate_all(
    df_samples, tc_all_filtered, tc_variables
)

# Keep only relevant columns
output_cols = ['Latitude', 'Longitude', 'Sample Date'] + tc_variables
terraclimate_df = terraclimate_df[output_cols]

print("\n✅ Climate values assigned!")
print(f"\nFinal shape: {terraclimate_df.shape}")
print(f"Expected: ({len(df_samples)}, {len(output_cols)})")

display(terraclimate_df.head(10))

Cell 9: Data Quality Checks


In [ ]:
print("="*80)
print("DATA QUALITY CHECKS")
print("="*80)

# Check for missing values
print("\n1. Missing Values:")
missing_counts = terraclimate_df[tc_variables].isna().sum()
missing_pct = (missing_counts / len(terraclimate_df) * 100).round(2)
missing_summary = pd.DataFrame({
    'Variable': missing_counts.index,
    'Missing Count': missing_counts.values,
    'Missing %': missing_pct.values
})
print(missing_summary.to_string(index=False))

# Check for unrealistic values
print("\n2. Value Ranges:")
print(terraclimate_df[tc_variables].describe())

# Check row count
print(f"\n3. Row Count Check:")
print(f"   Expected: {len(df_samples):,}")
print(f"   Actual:   {len(terraclimate_df):,}")
if len(terraclimate_df) == len(df_samples):
    print("   Row count matches!")
else:
    print("   Row count mismatch!")

# Check for duplicates
duplicates = terraclimate_df.duplicated(subset=['Latitude', 'Longitude', 'Sample Date']).sum()
print(f"\n4. Duplicate Check:")
print(f"   Duplicates: {duplicates}")
if duplicates == 0:
    print("   ✅ No duplicates!")
else:
    print("   ⚠️  Warning: Duplicates found!")

Cell 10: Save & Download files

In [ ]:
# Save to CSV
terraclimate_df.to_csv('terraclimate_features.csv', index=False)
print("="*80)
print("SAVE COMPLETE")
print("="*80)
print(f"\n✅ Saved: terraclimate_features.csv {terraclimate_df.shape}")

# Generate download link
print("\n" + "="*80)
print("DOWNLOAD LINK")
print("="*80)

csv = terraclimate_df.to_csv(index=False)
b64 = base64.b64encode(csv.encode()).decode()
display(HTML(f'<a href="data:file/csv;base64,{b64}" '
            f'download="terraclimate_features.csv">Download terraclimate_features.csv</a>'))

print("\n✅ TerraClimate extraction complete!")